## Score - 83908
- #### updated feature engineering in data preprocessing i.e. interaction columns, etc.

In [27]:
import numpy as np 
import pandas as pd  
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

In [28]:
pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/SampleSubmission.csv').sample(5)

,ID,TargetF1,TargetRAUC
838,ID_2892CFC2,0,0
696,ID_76A60DA1,0,0
1006,ID_013744F3,0,0
426,ID_8BEFF27C,0,0
454,ID_754FBABE,0,0


In [29]:
data_dict =pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/data_dictionary.csv')

In [30]:
df = pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/Train.csv')

In [31]:
df["is_climate_sensitive"].value_counts()

is_climate_sensitive
1    2047
0    1099
Name: count, dtype: int64

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3146 entries, 0 to 3145
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    3146 non-null   object 
 1   zone                  3146 non-null   object 
 2   gender                3146 non-null   object 
 3   deathdate             3146 non-null   object 
 4   age                   3146 non-null   float64
 5   avg_temperature       3146 non-null   float64
 6   max_temperature       3146 non-null   float64
 7   min_temperature       3146 non-null   float64
 8   precipitation         3146 non-null   float64
 9   latitude              3146 non-null   float64
 10  longitude             3146 non-null   float64
 11  location              3146 non-null   object 
 12  is_climate_sensitive  3146 non-null   int64  
dtypes: float64(7), int64(1), object(5)
memory usage: 319.6+ KB


In [33]:
prec_quant_75 = df["precipitation"].quantile(.75)
prec_quant_90 = df["precipitation"].quantile(.90)

In [34]:
def preprocessor(
    data,
    fit=True,
    target=None,
    te_maps=None,
    maxloc=1,
    prec_quant_75=prec_quant_75,
    prec_quant_90=prec_quant_90
):
    data = data.copy()

    data["zone"] = (data["zone"] == "Peri_urban").astype(int)
    data["gender"] = (data["gender"] == "Male").astype(int)

    data["age"] = data["age"].astype(int)

    data["age_group"] = pd.cut(
        data["age"],
        bins=[-1, 10, 20, 30, 40, 50, 60, 70, 80, 90, np.inf],
        labels=False
    ) + 1

    data["is_young"] = (data["age"] <= 20).astype(int)

    data["is_rain"] = (data["precipitation"] > 0).astype(int)

    data["is_heavy_rain"] = (
        data["precipitation"] > prec_quant_75
    ).astype(int)

    data["is_extreme_rain"] = (
        data["precipitation"] > prec_quant_90
    ).astype(int)

    data["precip_log"] = np.log1p(data["precipitation"])

    data["precip_per_temp"] = (
        data["precipitation"] /
        (abs(data["avg_temperature"]) + 1)
    )

    data["temp_rain_interaction"] = (
        data["avg_temperature"] *
        data["precipitation"]
    )

    data["temp_range"] = (
        data["max_temperature"] -
        data["min_temperature"]
    )

    data["avg_temp_position"] = (
        (data["avg_temperature"] - data["min_temperature"]) /
        (data["max_temperature"] - data["min_temperature"] + 1e-6)
    )

    loc_cols = [f"location_{i+1}" for i in range(maxloc)]

    location_split = data["location"].str.split(",", expand=True)

    for i, col in enumerate(loc_cols):
        if i < location_split.shape[1]:
            data[col] = location_split[i].str.strip()
        else:
            data[col] = "Unknown"

    data["deathdate"] = pd.to_datetime(
        data["deathdate"],
        errors="coerce"
    )

    data["date"] = data["deathdate"].dt.day
    data["month"] = data["deathdate"].dt.month
    data["year"] = data["deathdate"].dt.year
    data["day_of_week"] = data["deathdate"].dt.dayofweek
    data["is_weekend"] = (data["day_of_week"] >= 5).astype(int)
    data["quarter"] = data["deathdate"].dt.quarter
    data["day_of_year"] = data["deathdate"].dt.dayofyear
    data["week_of_year"] = (
        data["deathdate"].dt.isocalendar().week.astype(int)
    )

    te_cols = ["zone", "gender", "age_group"] + loc_cols

    if fit:
        if target is None:
            raise ValueError("target must be provided when fit=True")

        global_mean = data[target].mean()
        te_maps = {}

        for col in te_cols:
            stats = (
                pd.DataFrame({
                    "feature": data[col],
                    "target": data[target]
                })
                .groupby("feature")["target"]
                .agg(["mean", "count"])
            )

            smoothing = 10

            stats["encoded"] = (
                stats["count"] * stats["mean"]
                + smoothing * global_mean
            ) / (
                stats["count"] + smoothing
            )

            te_maps[col] = {
                "mapping": stats["encoded"].to_dict(),
                "global_mean": global_mean
            }

            data[f"{col}_te"] = (
                data[col]
                .map(te_maps[col]["mapping"])
                .fillna(global_mean)
            )

    else:
        if te_maps is None:
            raise ValueError("te_maps must be provided when fit=False")

        for col in te_cols:
            data[f"{col}_te"] = (
                data[col]
                .map(te_maps[col]["mapping"])
                .fillna(te_maps[col]["global_mean"])
            )

    for col in loc_cols:
        data[f"young_{col}"] = (
            data[f"{col}_te"] * data["is_young"]
        )

    data["zone_young"] = (
        data["zone_te"] * data["is_young"]
    )

    data["zone_gender"] = (
        data["zone_te"] * data["gender_te"]
    )

    data["gender_young"] = (
        data["gender_te"] * data["is_young"]
    )

    data["extreme_rain_latitude"] = (
        data["is_extreme_rain"] * data["latitude"]
    )

    data["extreme_rain_longitude"] = (
        data["is_extreme_rain"] * data["longitude"]
    )

    data["latitude_gender"] = (
        data["latitude"] * data["gender_te"]
    )

    data["longitude_gender"] = (
        data["longitude"] * data["gender_te"]
    )

    data["latitude_zone"] = (
        data["latitude"] * data["zone_te"]
    )

    data["longitude_zone"] = (
        data["longitude"] * data["zone_te"]
    )

    data["latitude_longitude"] = (
        data["latitude"] * data["longitude"]
    )

    data["latitude_longitude_gender"] = (
        data["latitude"] *
        data["longitude"] *
        data["gender_te"]
    )

    data["latitude_longitude_zone"] = (
        data["latitude"] *
        data["longitude"] *
        data["zone_te"]
    )

    data.drop(
        columns=["deathdate", "location"] + loc_cols,
        inplace=True
    )

    if fit:
        return data, te_maps

    return data

In [35]:
df1, enc_dict = preprocessor(df, fit=True, target='is_climate_sensitive')

In [36]:
X = df1.drop(columns=["is_climate_sensitive", "ID"])
y = df1["is_climate_sensitive"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [48]:
models = {

    "Gradient Boosting": GradientBoostingClassifier(
        learning_rate=0.01,
        min_samples_leaf=5,
        min_samples_split=10,
        n_estimators=300,
        random_state=42,
        subsample=0.9
    ),

    "Extra Trees": ExtraTreesClassifier(
        max_depth=5,
        max_features=None,
        min_samples_leaf=2,
        min_samples_split=5,
        n_estimators=500,
        n_jobs=-1,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        max_depth=5,
        max_features=None,
        min_samples_split=10,
        n_estimators=700,
        n_jobs=-1,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        colsample_bytree=1.0,
        eval_metric="logloss",
        gamma=0.1,
        learning_rate=0.01,
        max_depth=4,
        min_child_weight=3,
        n_estimators=300,
        n_jobs=-1,
        random_state=42
    )
}

tuned_models = {}
results = []

for name, model in models.items():

    model.fit(X_train, y_train)
    tuned_models[name] = model

    train_prob = model.predict_proba(X_train)[:, 1]
    train_pred = (train_prob >= 0.5).astype(int)

    train_f1 = f1_score(y_train, train_pred)
    train_auc = roc_auc_score(y_train, train_prob)
    train_score = 0.6 * train_f1 + 0.4 * train_auc

    test_prob = model.predict_proba(X_test)[:, 1]
    test_pred = (test_prob >= 0.5).astype(int)

    test_f1 = f1_score(y_test, test_pred)
    test_auc = roc_auc_score(y_test, test_prob)
    test_score = 0.6 * test_f1 + 0.4 * test_auc

    results.append({
        "Model": name,

        "Train F1": train_f1,
        "Train ROC-AUC": train_auc,
        "Train Final Score": train_score,

        "Test F1": test_f1,
        "Test ROC-AUC": test_auc,
        "Test Final Score": test_score,

        "Parameters": model.get_params()
    })

results_df = pd.DataFrame(results).sort_values(
    "Test Final Score",
    ascending=False
).reset_index(drop=True)

print(results_df)

               Model  Train F1  Train ROC-AUC  Train Final Score   Test F1  \
0  Gradient Boosting  0.838558       0.865303           0.849256  0.831615   
1        Extra Trees  0.826866       0.862884           0.841273  0.821596   
2            XGBoost  0.856635       0.885293           0.868098  0.821759   
3      Random Forest  0.863909       0.892726           0.875436  0.816803   

   Test ROC-AUC  Test Final Score  \
0      0.837783          0.834082   
1      0.834523          0.826767   
2      0.832483          0.826049   
3      0.831264          0.822587   

                                          Parameters  
0  {'ccp_alpha': 0.0, 'criterion': 'friedman_mse'...  
1  {'bootstrap': False, 'ccp_alpha': 0.0, 'class_...  
2  {'objective': 'binary:logistic', 'base_score':...  
3  {'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...  


In [39]:
tf = pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/Test.csv')

In [40]:
tf1 = preprocessor(tf, fit=False, te_maps=enc_dict, maxloc=1)

In [41]:
tf1.head()

,ID,zone,gender,age,avg_temperature,max_temperature,min_temperature,precipitation,latitude,longitude,...,gender_young,extreme_rain_latitude,extreme_rain_longitude,latitude_gender,longitude_gender,latitude_zone,longitude_zone,latitude_longitude,latitude_longitude_gender,latitude_longitude_zone
0,ID_E760D84B,0,0,75,21.224387,24.632729,18.145633,4.295335,0.616667,33.500000,...,0.000000,0.616667,33.500000,0.414039,22.492372,0.404943,21.998221,20.658344,13.870303,13.565577
1,ID_6EDEA907,0,0,50,21.708596,25.184528,18.285274,1.376196,0.725422,33.098845,...,0.000000,0.000000,0.000000,0.487058,22.223031,0.476358,21.734797,24.010617,16.121067,15.766891
2,ID_B9FFC8D8,0,0,76,21.371149,24.584523,18.866490,2.336643,0.603194,33.542370,...,0.000000,0.000000,0.000000,0.404993,22.520819,0.396095,22.026043,20.232543,13.584414,13.285968
3,ID_74C6C94E,1,0,90,21.341990,25.188662,17.998317,4.703071,-0.590211,30.056939,...,0.000000,-0.590211,30.056939,-0.396276,20.180652,-0.378107,19.255396,-17.739942,-11.910847,-11.364751
4,ID_0E02825D,0,1,8,19.710391,22.905167,18.082628,4.066863,0.603194,33.542370,...,0.631726,0.603194,33.542370,0.381053,21.189590,0.396095,22.026043,20.232543,12.781425,13.285968


In [42]:
df1.describe()

,zone,gender,age,avg_temperature,max_temperature,min_temperature,precipitation,latitude,longitude,is_climate_sensitive,...,gender_young,extreme_rain_latitude,extreme_rain_longitude,latitude_gender,longitude_gender,latitude_zone,longitude_zone,latitude_longitude,latitude_longitude_gender,latitude_longitude_zone
count,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,...,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000
mean,0.373172,0.522886,24.243484,22.066541,26.206879,18.514193,1.519907,0.630242,33.407442,0.650668,...,0.399394,0.063826,3.343360,0.410102,21.736643,0.410132,21.737159,21.017325,13.675821,13.676623
std,0.483724,0.499555,31.428833,1.095122,1.691347,0.891524,1.621088,0.172728,0.579933,0.476835,...,0.317241,0.199113,10.026484,0.113527,0.754285,0.113624,0.441033,5.499285,3.615275,3.616364
min,0.000000,0.000000,0.000000,18.852803,20.884623,15.018778,0.000000,0.089819,30.782817,0.000000,...,0.000000,0.000000,0.000000,0.056741,19.446308,0.058981,20.213946,2.913615,1.840607,1.913264
25%,0.000000,0.000000,0.000000,21.319885,25.260527,17.954814,0.286814,0.564758,33.473515,0.000000,...,0.000000,0.000000,0.000000,0.365934,21.151875,0.370856,21.445554,18.902508,12.274663,12.412583
50%,0.000000,1.000000,3.000000,21.931299,26.081829,18.510478,1.034344,0.591961,33.483544,1.000000,...,0.631726,0.000000,0.000000,0.384340,21.198698,0.386642,21.978149,19.917663,12.868431,12.942575
75%,1.000000,1.000000,49.000000,22.652860,27.015877,19.040655,2.222345,0.608396,33.519695,1.000000,...,0.671414,0.000000,0.000000,0.406286,22.480734,0.389757,22.011154,20.370734,13.600690,13.050117
max,1.000000,1.000000,110.000000,27.012223,34.080947,21.849602,16.606523,1.187892,34.226647,1.000000,...,0.671414,1.187892,34.226647,0.797567,22.980253,0.780045,22.475383,36.702698,24.642708,24.101315


In [43]:
def make_submission(
    test_data,
    model,
    id_col="ID"
):
    ids = test_data.pop(id_col)

    pred_binary = model.predict(test_data)
    pred_prob = model.predict_proba(test_data)[:, 1]
    submission = pd.DataFrame({
        "ID": ids,
        "TargetF1": pred_binary.astype(int),
        "TargetRAUC": pred_prob
    })

    return submission

In [49]:
best_name = results_df.iloc[0]["Model"]
best_model = tuned_models[best_name]

X_full = df1.drop(columns=["is_climate_sensitive", "ID"])
y_full = df1["is_climate_sensitive"]

best_model.fit(X_full, y_full)

X_submission = tf1.drop(columns=["ID"])
ids = tf1["ID"].copy()

pred_prob = best_model.predict_proba(X_submission)[:, 1]
pred_binary = (pred_prob >= 0.5).astype(int)

submission = pd.DataFrame({
    "ID": ids,
    "TargetF1": pred_binary,
    "TargetRAUC": pred_prob
})

submission.to_csv("climate-risk-fe-updated.csv", index=False)

print("Best model:", best_name)
print(submission.head())

Best model: Gradient Boosting
            ID  TargetF1  TargetRAUC
0  ID_E760D84B         1    0.517102
1  ID_6EDEA907         1    0.662615
2  ID_B9FFC8D8         0    0.453353
3  ID_74C6C94E         0    0.479632
4  ID_0E02825D         1    0.705831
